# MOSS-Image · T4 ComfyUI 프론트

**노트북 버전 2026-11-10.1**

[실험 브랜치의 Colab 노트북](https://colab.research.google.com/github/HisameOgasahara/tmp_moss/blob/codex/comfyui-latest/MOSS_T4_ComfyUI.ipynb)

**T4 GPU 선택 → 1~3번 실행 → ComfyUI 열기 → 워크플로 불러오기 → Run.**

- N1-lora8+spectrum: INT8 본체 / W4A8 인코더 / 8스텝 LoRA / Spectrum Manual.
- 기본값: 1024×1024 · batch 1 · Euler · sgm_uniform · 8스텝 · CFG 1.
- 프롬프트·해상도·시드·생성 설정은 ComfyUI 화면에서 변경합니다. 생성 이미지는 Save Image 노드에서 저장합니다.
- 워크플로는 서버의 Workflows 목록에 등록되며 4번 셀에서도 JSON을 내려받을 수 있습니다.
- 같은 Colab 런타임의 기존 `/content/moss_t4` 모델 캐시를 재사용합니다.

터널 주소를 아는 사람은 ComfyUI에 접속할 수 있습니다. 사용 후 실행 중인 3번 셀의 중지 버튼을 누르면 연결이 종료됩니다.


In [ ]:
#@title 1. T4 · ComfyUI · Spectrum 설치
import sys, os, json, time, re, shutil, hashlib, subprocess, urllib.request, urllib.error
from pathlib import Path
from IPython.display import display, HTML

ROOT = Path('/content/moss_t4')
COMFY = ROOT/'ComfyUI'
VENV = ROOT/'venv'
PY = str(VENV/'bin/python')
PORT = 8188
LOCAL_URL = f'http://127.0.0.1:{PORT}'
SPECTRUM_REV = 'd3f7e243ea2db6cc3ff26f42b935b167f938b599'
MODEL_REV = '24147db865fcdc207270f844777587223fed8d6e'
LORA_REV = 'f95df83b5e90ae91ea412434b3f9b7f1c8c6314f'
ROOT.mkdir(parents=True,exist_ok=True)
INSTALL_LOG = ROOT/'frontend-install.log'
INSTALL_LOG.write_text('',encoding='utf-8')
gpu = subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],text=True).strip()
if 'T4' not in gpu:
    raise RuntimeError(f'T4 GPU를 선택하세요. 현재 GPU: {gpu}')
print('GPU:',gpu)

def run(command, **kwargs):
    command = [str(x) for x in command]
    lines = []
    with INSTALL_LOG.open('a',encoding='utf-8') as log:
        log.write('> '+' '.join(command)+chr(10))
        with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,bufsize=1,**kwargs) as process:
            for line in process.stdout:
                print(line,end='',flush=True)
                log.write(line)
                log.flush()
                lines.append(line)
            code = process.wait()
    if code:
        raise RuntimeError(f'설치 실패 (exit {code}) · {INSTALL_LOG}'+chr(10)+''.join(lines[-40:]))

def fetch(url):
    request = urllib.request.Request(url,headers={'User-Agent':'moss-t4-comfyui'})
    with urllib.request.urlopen(request,timeout=120) as response:
        return response.read()

if shutil.which('uv') is None:
    run([sys.executable,'-m','pip','install','-q','uv>=0.9.27'])
if '--excludes' not in subprocess.check_output(['uv','pip','install','--help'],text=True):
    run([sys.executable,'-m','pip','install','-q','--upgrade','uv>=0.9.27'])
if not Path(PY).exists():
    run(['uv','venv','--system-site-packages','--python',sys.executable,VENV])
if not (COMFY/'.git').exists():
    run(['git','clone','--quiet','https://github.com/Comfy-Org/ComfyUI.git',COMFY])
COMFY_TAG = json.loads(fetch('https://api.github.com/repos/Comfy-Org/ComfyUI/releases/latest'))['tag_name']
run(['git','-C',COMFY,'fetch','--quiet','--no-tags','origin',f'refs/tags/{COMFY_TAG}'])
run(['git','-C',COMFY,'checkout','--quiet','FETCH_HEAD'])
COMFY_REV = subprocess.check_output(['git','-C',str(COMFY),'rev-parse','HEAD'], text=True).strip()
# Colab의 기존 torch/CUDA 패키지를 상속합니다.
runtime_code = """
import importlib.metadata as m, json
packages = {}
for dist in m.distributions():
    name = dist.metadata['Name'].lower().replace('_','-')
    if name in ('torch','torchvision','torchaudio','triton') or name.startswith('nvidia-'):
        packages[name] = dist.version
print(json.dumps(packages))
"""
runtime_packages = json.loads(subprocess.check_output([sys.executable,'-c',runtime_code],text=True))
excludes = ROOT/'colab-runtime-excludes.txt'
excludes.write_text(chr(10).join(sorted(runtime_packages))+chr(10),encoding='utf-8')
run(['uv','pip','install','--python',PY,'--excludes',excludes,'-r',COMFY/'requirements.txt','huggingface_hub'])
addon = COMFY/'custom_nodes'/'ComfyUI-Spectrum-MOSS'
if not (addon/'.git').exists():
    run(['git','clone','--quiet','https://github.com/Rkkss/ComfyUI-Spectrum-MOSS.git',addon])
run(['git','-C',addon,'checkout','--quiet',SPECTRUM_REV])
print('ComfyUI · Spectrum 설치 완료')


In [ ]:
#@title 2. 선정 모델 · 워크플로 · cloudflared 다운로드
from concurrent.futures import ThreadPoolExecutor

DOWNLOAD_WORKERS = 3
resources = [
    dict(repo='Comfy-Org/MOSS-Image',revision=MODEL_REV,
         filename='diffusion_models/moss_image_sft_int8_convrot.safetensors',
         target=str(COMFY/'models/diffusion_models/moss_image_sft_int8_convrot.safetensors')),
    dict(repo='Comfy-Org/MOSS-Image',revision=MODEL_REV,
         filename='text_encoders/qwen3.5_9b_w4a8_gptq.safetensors',
         target=str(COMFY/'models/text_encoders/qwen3.5_9b_w4a8_gptq.safetensors')),
    dict(repo='Comfy-Org/MOSS-Image',revision=MODEL_REV,
         filename='vae/flux2-vae_small.safetensors',
         target=str(COMFY/'models/vae/flux2-vae_small.safetensors')),
    dict(repo='lodestones/MOSS-Image-SFT-8-steps-dmd',revision=LORA_REV,filename='lora.safetensors',
         target=str(COMFY/'models/loras/moss_sft_8step_lora.safetensors')),
]
manifest = ROOT/'frontend-downloads.json'
manifest.write_text(json.dumps(resources,indent=2),encoding='utf-8')
download_code = """
import json, sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from huggingface_hub import hf_hub_download

def download_hf_model(item):
    print('Hugging Face 다운로드 요청:',item['filename'],flush=True)
    source = hf_hub_download(item['repo'],item['filename'],revision=item['revision'],cache_dir=sys.argv[2])
    target = Path(item['target'])
    target.parent.mkdir(parents=True,exist_ok=True)
    if target.is_symlink():
        target.unlink()
    if not target.exists():
        target.symlink_to(source)
    print('Hugging Face 파일 준비 완료:',item['filename'],flush=True)

items = json.loads(Path(sys.argv[1]).read_text())
with ThreadPoolExecutor(max_workers=int(sys.argv[3])) as executor:
    list(executor.map(download_hf_model, items))
"""
run([PY,'-c',download_code,manifest,ROOT/'hf-cache',DOWNLOAD_WORKERS])

from google.colab import userdata
from tempfile import NamedTemporaryFile

CIVITAI_LORAS = [
    ('3437656?fileId=3334919', 'moss_style_nai5_3437656.safetensors',
     '7166832f2c1146c45da4e96d6f631f1ed187f6b544e93d857f6b81be800a6586'),
    ('3442141?fileId=3340383', 'Munji_MOSSV1.safetensors',
     '1fa373f82866b357372b1ff055eabf67cad14bccb5179eb45401c18444986bdd'),
    ('3453015?fileId=3353385', 'nai3_v1_step2000 (2).safetensors',
     'f8882ee058af9190da34fd72c0796c7154833d5361ee2b6eb603900ea6cae2ab'),
]

try:
    civitai_token = userdata.get('CIVITAI_API_TOKEN')
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    civitai_token = None

def download_civitai_lora(version, filename, expected_sha256, token):
    target = COMFY/'models'/'loras'/filename
    if target.is_file():
        with target.open('rb') as cached:
            if hashlib.file_digest(cached, 'sha256').hexdigest() == expected_sha256:
                print(f'Civitai 캐시 재사용: {filename}',flush=True)
                return True
    if not token:
        print(f'Civitai 다운로드 건너뜀: {filename} · CIVITAI_API_TOKEN 없음 또는 액세스 미허용',flush=True)
        return False

    request = urllib.request.Request(f'https://civitai.com/api/download/models/{version}')
    # Only send the token to Civitai, not its download CDN.
    request.add_unredirected_header('Authorization', f'Bearer {token}')
    temporary_path = None
    print(f'Civitai 다운로드 시작: {filename}',flush=True)
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            with NamedTemporaryFile(dir=target.parent, suffix='.part', delete=False) as temporary:
                temporary_path = Path(temporary.name)
                shutil.copyfileobj(response, temporary)
        with temporary_path.open('rb') as downloaded:
            if hashlib.file_digest(downloaded, 'sha256').hexdigest() != expected_sha256:
                print(f'Civitai 다운로드 실패: {filename} · SHA256 불일치',flush=True)
                return False
        temporary_path.replace(target)
        print(f'Civitai 다운로드 완료: {filename}',flush=True)
        return True
    except urllib.error.HTTPError as error:
        print(f'Civitai 다운로드 실패: {filename} · HTTP {error.code}',flush=True)
        return False
    except (urllib.error.URLError, TimeoutError, OSError) as error:
        print(f'Civitai 다운로드 실패: {filename} · {type(error).__name__}',flush=True)
        return False
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)

print(f'Civitai LoRA {len(CIVITAI_LORAS)}개 확인 시작',flush=True)
with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    downloads = [executor.submit(download_civitai_lora, *lora, civitai_token) for lora in CIVITAI_LORAS]
    civitai_ready = sum(download.result() for download in downloads)
print(f'Civitai LoRA 준비 완료: {civitai_ready}/{len(CIVITAI_LORAS)}개',flush=True)


WORKFLOW_JSON = '{\n  "last_node_id": 11,\n  "last_link_id": 11,\n  "nodes": [\n    {\n      "id": 1,\n      "type": "UNETLoader",\n      "title": "MOSS INT8 본체",\n      "pos": [\n        40,\n        40\n      ],\n      "size": [\n        340,\n        110\n      ],\n      "flags": {},\n      "order": 0,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "MODEL",\n          "type": "MODEL",\n          "links": [\n            1\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "UNETLoader"\n      },\n      "widgets_values": [\n        "moss_image_sft_int8_convrot.safetensors",\n        "default"\n      ]\n    },\n    {\n      "id": 2,\n      "type": "LoraLoaderModelOnly",\n      "title": "8스텝 LoRA",\n      "pos": [\n        40,\n        210\n      ],\n      "size": [\n        340,\n        130\n      ],\n      "flags": {},\n      "order": 1,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 1\n        }\n      ],\n      "outputs": [\n        {\n          "name": "MODEL",\n          "type": "MODEL",\n          "links": [\n            2\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "LoraLoaderModelOnly"\n      },\n      "widgets_values": [\n        "moss_sft_8step_lora.safetensors",\n        1.0\n      ]\n    },\n    {\n      "id": 3,\n      "type": "SpectrumMOSSImageManual",\n      "title": "Spectrum · 검증 설정",\n      "pos": [\n        440,\n        40\n      ],\n      "size": [\n        350,\n        490\n      ],\n      "flags": {},\n      "order": 2,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 2\n        }\n      ],\n      "outputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "links": [\n            3\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "SpectrumMOSSImageManual"\n      },\n      "widgets_values": [\n        2,\n        0,\n        2.0,\n        0.75,\n        2,\n        2,\n        1,\n        0.1,\n        0.5,\n        "main_device",\n        true,\n        false,\n        true\n      ]\n    },\n    {\n      "id": 4,\n      "type": "CLIPLoader",\n      "title": "Qwen W4A8 인코더",\n      "pos": [\n        40,\n        420\n      ],\n      "size": [\n        340,\n        130\n      ],\n      "flags": {},\n      "order": 3,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "CLIP",\n          "type": "CLIP",\n          "links": [\n            4,\n            5\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "CLIPLoader"\n      },\n      "widgets_values": [\n        "qwen3.5_9b_w4a8_gptq.safetensors",\n        "moss_image",\n        "default"\n      ]\n    },\n    {\n      "id": 5,\n      "type": "VAELoader",\n      "title": "FLUX.2 small VAE",\n      "pos": [\n        840,\n        670\n      ],\n      "size": [\n        330,\n        80\n      ],\n      "flags": {},\n      "order": 4,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "VAE",\n          "type": "VAE",\n          "links": [\n            10\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "VAELoader"\n      },\n      "widgets_values": [\n        "flux2-vae_small.safetensors"\n      ]\n    },\n    {\n      "id": 6,\n      "type": "CLIPTextEncode",\n      "title": "양성 프롬프트",\n      "pos": [\n        440,\n        590\n      ],\n      "size": [\n        350,\n        180\n      ],\n      "flags": {},\n      "order": 5,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "clip",\n          "type": "CLIP",\n          "link": 4\n        }\n      ],\n      "outputs": [\n        {\n          "name": "CONDITIONING",\n          "type": "CONDITIONING",\n          "links": [\n            6\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "CLIPTextEncode"\n      },\n      "widgets_values": [\n        "A studio photograph of a red ceramic teapot on a wooden table, soft window light."\n      ]\n    },\n    {\n      "id": 7,\n      "type": "CLIPTextEncode",\n      "title": "음성 프롬프트",\n      "pos": [\n        440,\n        840\n      ],\n      "size": [\n        350,\n        120\n      ],\n      "flags": {},\n      "order": 6,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "clip",\n          "type": "CLIP",\n          "link": 5\n        }\n      ],\n      "outputs": [\n        {\n          "name": "CONDITIONING",\n          "type": "CONDITIONING",\n          "links": [\n            7\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "CLIPTextEncode"\n      },\n      "widgets_values": [\n        ""\n      ]\n    },\n    {\n      "id": 8,\n      "type": "EmptyFlux2LatentImage",\n      "title": "해상도 · batch 1",\n      "pos": [\n        840,\n        450\n      ],\n      "size": [\n        330,\n        120\n      ],\n      "flags": {},\n      "order": 7,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "LATENT",\n          "type": "LATENT",\n          "links": [\n            8\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "EmptyFlux2LatentImage"\n      },\n      "widgets_values": [\n        1024,\n        1024,\n        1\n      ]\n    },\n    {\n      "id": 9,\n      "type": "KSampler",\n      "title": "Euler · 8스텝 · CFG 1",\n      "pos": [\n        840,\n        40\n      ],\n      "size": [\n        330,\n        320\n      ],\n      "flags": {},\n      "order": 8,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 3\n        },\n        {\n          "name": "positive",\n          "type": "CONDITIONING",\n          "link": 6\n        },\n        {\n          "name": "negative",\n          "type": "CONDITIONING",\n          "link": 7\n        },\n        {\n          "name": "latent_image",\n          "type": "LATENT",\n          "link": 8\n        }\n      ],\n      "outputs": [\n        {\n          "name": "LATENT",\n          "type": "LATENT",\n          "links": [\n            9\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "KSampler"\n      },\n      "widgets_values": [\n        42,\n        "randomize",\n        8,\n        1.0,\n        "euler",\n        "sgm_uniform",\n        1.0\n      ]\n    },\n    {\n      "id": 10,\n      "type": "VAEDecode",\n      "title": "VAE 디코딩",\n      "pos": [\n        1240,\n        40\n      ],\n      "size": [\n        230,\n        100\n      ],\n      "flags": {},\n      "order": 9,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "samples",\n          "type": "LATENT",\n          "link": 9\n        },\n        {\n          "name": "vae",\n          "type": "VAE",\n          "link": 10\n        }\n      ],\n      "outputs": [\n        {\n          "name": "IMAGE",\n          "type": "IMAGE",\n          "links": [\n            11\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "VAEDecode"\n      },\n      "widgets_values": []\n    },\n    {\n      "id": 11,\n      "type": "SaveImage",\n      "title": "생성 이미지",\n      "pos": [\n        1540,\n        40\n      ],\n      "size": [\n        470,\n        500\n      ],\n      "flags": {},\n      "order": 10,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "images",\n          "type": "IMAGE",\n          "link": 11\n        }\n      ],\n      "outputs": [\n        {\n          "name": "images",\n          "type": "IMAGE",\n          "links": null,\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "SaveImage"\n      },\n      "widgets_values": [\n        "MOSS_T4"\n      ]\n    }\n  ],\n  "links": [\n    [\n      1,\n      1,\n      0,\n      2,\n      0,\n      "MODEL"\n    ],\n    [\n      2,\n      2,\n      0,\n      3,\n      0,\n      "MODEL"\n    ],\n    [\n      3,\n      3,\n      0,\n      9,\n      0,\n      "MODEL"\n    ],\n    [\n      4,\n      4,\n      0,\n      6,\n      0,\n      "CLIP"\n    ],\n    [\n      5,\n      4,\n      0,\n      7,\n      0,\n      "CLIP"\n    ],\n    [\n      6,\n      6,\n      0,\n      9,\n      1,\n      "CONDITIONING"\n    ],\n    [\n      7,\n      7,\n      0,\n      9,\n      2,\n      "CONDITIONING"\n    ],\n    [\n      8,\n      8,\n      0,\n      9,\n      3,\n      "LATENT"\n    ],\n    [\n      9,\n      9,\n      0,\n      10,\n      0,\n      "LATENT"\n    ],\n    [\n      10,\n      5,\n      0,\n      10,\n      1,\n      "VAE"\n    ],\n    [\n      11,\n      10,\n      0,\n      11,\n      0,\n      "IMAGE"\n    ]\n  ],\n  "groups": [],\n  "config": {},\n  "extra": {\n    "ds": {\n      "scale": 0.7,\n      "offset": [\n        60,\n        80\n      ]\n    }\n  },\n  "version": 0.4\n}\n'
workflow = json.loads(WORKFLOW_JSON)
WORKFLOW_FILE = ROOT/'MOSS_T4_LoRA8_Spectrum.json'
WORKFLOW_FILE.write_text(WORKFLOW_JSON,encoding='utf-8')
workflow_library = COMFY/'user'/'default'/'workflows'/WORKFLOW_FILE.name
workflow_library.parent.mkdir(parents=True,exist_ok=True)
workflow_library.write_text(WORKFLOW_JSON,encoding='utf-8')

EXPERIMENTAL_WORKFLOWS = {
    'MOSS_T4_NAI5_Style.json': '{"last_node_id":11,"last_link_id":12,"nodes":[{"id":1,"type":"UNETLoader","title":"MOSS INT8 본체","pos":[40,40],"size":[340,110],"flags":{},"order":0,"mode":0,"inputs":[],"outputs":[{"name":"MODEL","type":"MODEL","links":[1],"slot_index":0}],"properties":{"Node name for S&R":"UNETLoader"},"widgets_values":["moss_image_sft_int8_convrot.safetensors","default"]},{"id":2,"type":"LoraLoader","title":"MOSS_T4_NAI5_Style","pos":[40,210],"size":[340,130],"flags":{},"order":1,"mode":0,"inputs":[{"name":"model","type":"MODEL","link":1},{"name":"clip","type":"CLIP","link":12}],"outputs":[{"name":"MODEL","type":"MODEL","links":[2],"slot_index":0},{"name":"CLIP","type":"CLIP","links":[4,5],"slot_index":1}],"properties":{"Node name for S&R":"LoraLoader"},"widgets_values":["moss_style_nai5_3437656.safetensors",1.0,1.0]},{"id":3,"type":"SkipLayerGuidanceDiTSimple","title":"Skip Layer Guidance","pos":[440,40],"size":[350,180],"flags":{},"order":2,"mode":0,"inputs":[{"name":"model","type":"MODEL","link":2}],"outputs":[{"name":"model","type":"MODEL","links":[3],"slot_index":0}],"properties":{"Node name for S&R":"SkipLayerGuidanceDiTSimple"},"widgets_values":["","22",0.0,0.5]},{"id":4,"type":"CLIPLoader","title":"Qwen W4A8 인코더","pos":[40,420],"size":[340,130],"flags":{},"order":3,"mode":0,"inputs":[],"outputs":[{"name":"CLIP","type":"CLIP","links":[12],"slot_index":0}],"properties":{"Node name for S&R":"CLIPLoader"},"widgets_values":["qwen3.5_9b_w4a8_gptq.safetensors","moss_image","default"]},{"id":5,"type":"VAELoader","title":"FLUX.2 small VAE","pos":[840,670],"size":[330,80],"flags":{},"order":4,"mode":0,"inputs":[],"outputs":[{"name":"VAE","type":"VAE","links":[10],"slot_index":0}],"properties":{"Node name for S&R":"VAELoader"},"widgets_values":["flux2-vae_small.safetensors"]},{"id":6,"type":"CLIPTextEncode","title":"양성 프롬프트","pos":[440,590],"size":[350,180],"flags":{},"order":5,"mode":0,"inputs":[{"name":"clip","type":"CLIP","link":4}],"outputs":[{"name":"CONDITIONING","type":"CONDITIONING","links":[6],"slot_index":0}],"properties":{"Node name for S&R":"CLIPTextEncode"},"widgets_values":["@neoanime, An anime illustration of a traveler at a quiet railway station in the rain."]},{"id":7,"type":"CLIPTextEncode","title":"음성 프롬프트","pos":[440,840],"size":[350,120],"flags":{},"order":6,"mode":0,"inputs":[{"name":"clip","type":"CLIP","link":5}],"outputs":[{"name":"CONDITIONING","type":"CONDITIONING","links":[7],"slot_index":0}],"properties":{"Node name for S&R":"CLIPTextEncode"},"widgets_values":[""]},{"id":8,"type":"EmptyFlux2LatentImage","title":"해상도 · batch 1","pos":[840,450],"size":[330,120],"flags":{},"order":7,"mode":0,"inputs":[],"outputs":[{"name":"LATENT","type":"LATENT","links":[8],"slot_index":0}],"properties":{"Node name for S&R":"EmptyFlux2LatentImage"},"widgets_values":[816,1280,1]},{"id":9,"type":"KSampler","title":"Euler · 8스텝 · CFG 1","pos":[840,40],"size":[330,320],"flags":{},"order":8,"mode":0,"inputs":[{"name":"model","type":"MODEL","link":3},{"name":"positive","type":"CONDITIONING","link":6},{"name":"negative","type":"CONDITIONING","link":7},{"name":"latent_image","type":"LATENT","link":8}],"outputs":[{"name":"LATENT","type":"LATENT","links":[9],"slot_index":0}],"properties":{"Node name for S&R":"KSampler"},"widgets_values":[6765770955,"fixed",35,3.0,"er_sde","sgm_uniform",1.0]},{"id":10,"type":"VAEDecode","title":"VAE 디코딩","pos":[1240,40],"size":[230,100],"flags":{},"order":9,"mode":0,"inputs":[{"name":"samples","type":"LATENT","link":9},{"name":"vae","type":"VAE","link":10}],"outputs":[{"name":"IMAGE","type":"IMAGE","links":[11],"slot_index":0}],"properties":{"Node name for S&R":"VAEDecode"},"widgets_values":[]},{"id":11,"type":"SaveImage","title":"생성 이미지","pos":[1540,40],"size":[470,500],"flags":{},"order":10,"mode":0,"inputs":[{"name":"images","type":"IMAGE","link":11}],"outputs":[{"name":"images","type":"IMAGE","links":null,"slot_index":0}],"properties":{"Node name for S&R":"SaveImage"},"widgets_values":["MOSS_T4_NAI5_Style"]}],"links":[[1,1,0,2,0,"MODEL"],[2,2,0,3,0,"MODEL"],[3,3,0,9,0,"MODEL"],[4,2,1,6,0,"CLIP"],[5,2,1,7,0,"CLIP"],[6,6,0,9,1,"CONDITIONING"],[7,7,0,9,2,"CONDITIONING"],[8,8,0,9,3,"LATENT"],[9,9,0,10,0,"LATENT"],[10,5,0,10,1,"VAE"],[11,10,0,11,0,"IMAGE"],[12,4,0,2,1,"CLIP"]],"groups":[],"config":{},"extra":{"ds":{"scale":0.7,"offset":[60,80]}},"version":0.4}',
    'MOSS_T4_Munji_Style.json': '{"last_node_id":11,"last_link_id":12,"nodes":[{"id":1,"type":"UNETLoader","title":"MOSS INT8 본체","pos":[40,40],"size":[340,110],"flags":{},"order":0,"mode":0,"inputs":[],"outputs":[{"name":"MODEL","type":"MODEL","links":[1],"slot_index":0}],"properties":{"Node name for S&R":"UNETLoader"},"widgets_values":["moss_image_sft_int8_convrot.safetensors","default"]},{"id":2,"type":"LoraLoader","title":"MOSS_T4_Munji_Style","pos":[40,210],"size":[340,130],"flags":{},"order":1,"mode":0,"inputs":[{"name":"model","type":"MODEL","link":1},{"name":"clip","type":"CLIP","link":12}],"outputs":[{"name":"MODEL","type":"MODEL","links":[2],"slot_index":0},{"name":"CLIP","type":"CLIP","links":[4,5],"slot_index":1}],"properties":{"Node name for S&R":"LoraLoader"},"widgets_values":["Munji_MOSSV1.safetensors",1.0,1.0]},{"id":3,"type":"SkipLayerGuidanceDiTSimple","title":"Skip Layer Guidance","pos":[440,40],"size":[350,180],"flags":{},"order":2,"mode":0,"inputs":[{"name":"model","type":"MODEL","link":2}],"outputs":[{"name":"model","type":"MODEL","links":[3],"slot_index":0}],"properties":{"Node name for S&R":"SkipLayerGuidanceDiTSimple"},"widgets_values":["","22",0.0,1.0]},{"id":4,"type":"CLIPLoader","title":"Qwen W4A8 인코더","pos":[40,420],"size":[340,130],"flags":{},"order":3,"mode":0,"inputs":[],"outputs":[{"name":"CLIP","type":"CLIP","links":[12],"slot_index":0}],"properties":{"Node name for S&R":"CLIPLoader"},"widgets_values":["qwen3.5_9b_w4a8_gptq.safetensors","moss_image","default"]},{"id":5,"type":"VAELoader","title":"FLUX.2 small VAE","pos":[840,670],"size":[330,80],"flags":{},"order":4,"mode":0,"inputs":[],"outputs":[{"name":"VAE","type":"VAE","links":[10],"slot_index":0}],"properties":{"Node name for S&R":"VAELoader"},"widgets_values":["flux2-vae_small.safetensors"]},{"id":6,"type":"CLIPTextEncode","title":"양성 프롬프트","pos":[440,590],"size":[350,180],"flags":{},"order":5,"mode":0,"inputs":[{"name":"clip","type":"CLIP","link":4}],"outputs":[{"name":"CONDITIONING","type":"CONDITIONING","links":[6],"slot_index":0}],"properties":{"Node name for S&R":"CLIPTextEncode"},"widgets_values":["@muji_style, an anime portrait of a traveler at a quiet railway station in the rain"]},{"id":7,"type":"CLIPTextEncode","title":"음성 프롬프트","pos":[440,840],"size":[350,120],"flags":{},"order":6,"mode":0,"inputs":[{"name":"clip","type":"CLIP","link":5}],"outputs":[{"name":"CONDITIONING","type":"CONDITIONING","links":[7],"slot_index":0}],"properties":{"Node name for S&R":"CLIPTextEncode"},"widgets_values":[""]},{"id":8,"type":"EmptyFlux2LatentImage","title":"해상도 · batch 1","pos":[840,450],"size":[330,120],"flags":{},"order":7,"mode":0,"inputs":[],"outputs":[{"name":"LATENT","type":"LATENT","links":[8],"slot_index":0}],"properties":{"Node name for S&R":"EmptyFlux2LatentImage"},"widgets_values":[1024,1536,1]},{"id":9,"type":"KSampler","title":"Euler · 8스텝 · CFG 1","pos":[840,40],"size":[330,320],"flags":{},"order":8,"mode":0,"inputs":[{"name":"model","type":"MODEL","link":3},{"name":"positive","type":"CONDITIONING","link":6},{"name":"negative","type":"CONDITIONING","link":7},{"name":"latent_image","type":"LATENT","link":8}],"outputs":[{"name":"LATENT","type":"LATENT","links":[9],"slot_index":0}],"properties":{"Node name for S&R":"KSampler"},"widgets_values":[390702223096865,"fixed",30,3.0,"er_sde","beta",1.0]},{"id":10,"type":"VAEDecode","title":"VAE 디코딩","pos":[1240,40],"size":[230,100],"flags":{},"order":9,"mode":0,"inputs":[{"name":"samples","type":"LATENT","link":9},{"name":"vae","type":"VAE","link":10}],"outputs":[{"name":"IMAGE","type":"IMAGE","links":[11],"slot_index":0}],"properties":{"Node name for S&R":"VAEDecode"},"widgets_values":[]},{"id":11,"type":"SaveImage","title":"생성 이미지","pos":[1540,40],"size":[470,500],"flags":{},"order":10,"mode":0,"inputs":[{"name":"images","type":"IMAGE","link":11}],"outputs":[{"name":"images","type":"IMAGE","links":null,"slot_index":0}],"properties":{"Node name for S&R":"SaveImage"},"widgets_values":["MOSS_T4_Munji_Style"]}],"links":[[1,1,0,2,0,"MODEL"],[2,2,0,3,0,"MODEL"],[3,3,0,9,0,"MODEL"],[4,2,1,6,0,"CLIP"],[5,2,1,7,0,"CLIP"],[6,6,0,9,1,"CONDITIONING"],[7,7,0,9,2,"CONDITIONING"],[8,8,0,9,3,"LATENT"],[9,9,0,10,0,"LATENT"],[10,5,0,10,1,"VAE"],[11,10,0,11,0,"IMAGE"],[12,4,0,2,1,"CLIP"]],"groups":[],"config":{},"extra":{"ds":{"scale":0.7,"offset":[60,80]}},"version":0.4}',
    'MOSS_T4_FFstyle_Heal.json': '{"last_node_id":11,"last_link_id":12,"nodes":[{"id":1,"type":"UNETLoader","title":"MOSS INT8 본체","pos":[40,40],"size":[340,110],"flags":{},"order":0,"mode":0,"inputs":[],"outputs":[{"name":"MODEL","type":"MODEL","links":[1],"slot_index":0}],"properties":{"Node name for S&R":"UNETLoader"},"widgets_values":["moss_image_sft_int8_convrot.safetensors","default"]},{"id":2,"type":"LoraLoader","title":"MOSS_T4_FFstyle_Heal","pos":[40,210],"size":[340,130],"flags":{},"order":1,"mode":0,"inputs":[{"name":"model","type":"MODEL","link":1},{"name":"clip","type":"CLIP","link":12}],"outputs":[{"name":"MODEL","type":"MODEL","links":[3],"slot_index":0},{"name":"CLIP","type":"CLIP","links":[4,5],"slot_index":1}],"properties":{"Node name for S&R":"LoraLoader"},"widgets_values":["nai3_v1_step2000 (2).safetensors",1.0,1.0]},{"id":4,"type":"CLIPLoader","title":"Qwen W4A8 인코더","pos":[40,420],"size":[340,130],"flags":{},"order":3,"mode":0,"inputs":[],"outputs":[{"name":"CLIP","type":"CLIP","links":[12],"slot_index":0}],"properties":{"Node name for S&R":"CLIPLoader"},"widgets_values":["qwen3.5_9b_w4a8_gptq.safetensors","moss_image","default"]},{"id":5,"type":"VAELoader","title":"FLUX.2 small VAE","pos":[840,670],"size":[330,80],"flags":{},"order":4,"mode":0,"inputs":[],"outputs":[{"name":"VAE","type":"VAE","links":[10],"slot_index":0}],"properties":{"Node name for S&R":"VAELoader"},"widgets_values":["flux2-vae_small.safetensors"]},{"id":6,"type":"CLIPTextEncode","title":"양성 프롬프트","pos":[440,590],"size":[350,180],"flags":{},"order":5,"mode":0,"inputs":[{"name":"clip","type":"CLIP","link":4}],"outputs":[{"name":"CONDITIONING","type":"CONDITIONING","links":[6],"slot_index":0}],"properties":{"Node name for S&R":"CLIPTextEncode"},"widgets_values":["An anime illustration of a traveler at a quiet railway station in the rain."]},{"id":7,"type":"CLIPTextEncode","title":"음성 프롬프트","pos":[440,840],"size":[350,120],"flags":{},"order":6,"mode":0,"inputs":[{"name":"clip","type":"CLIP","link":5}],"outputs":[{"name":"CONDITIONING","type":"CONDITIONING","links":[7],"slot_index":0}],"properties":{"Node name for S&R":"CLIPTextEncode"},"widgets_values":[""]},{"id":8,"type":"EmptyFlux2LatentImage","title":"해상도 · batch 1","pos":[840,450],"size":[330,120],"flags":{},"order":7,"mode":0,"inputs":[],"outputs":[{"name":"LATENT","type":"LATENT","links":[8],"slot_index":0}],"properties":{"Node name for S&R":"EmptyFlux2LatentImage"},"widgets_values":[1280,1920,1]},{"id":9,"type":"KSampler","title":"Euler · 8스텝 · CFG 1","pos":[840,40],"size":[330,320],"flags":{},"order":8,"mode":0,"inputs":[{"name":"model","type":"MODEL","link":3},{"name":"positive","type":"CONDITIONING","link":6},{"name":"negative","type":"CONDITIONING","link":7},{"name":"latent_image","type":"LATENT","link":8}],"outputs":[{"name":"LATENT","type":"LATENT","links":[9],"slot_index":0}],"properties":{"Node name for S&R":"KSampler"},"widgets_values":[296288883306814,"fixed",25,4.0,"euler","simple",1.0]},{"id":10,"type":"VAEDecode","title":"VAE 디코딩","pos":[1240,40],"size":[230,100],"flags":{},"order":9,"mode":0,"inputs":[{"name":"samples","type":"LATENT","link":9},{"name":"vae","type":"VAE","link":10}],"outputs":[{"name":"IMAGE","type":"IMAGE","links":[11],"slot_index":0}],"properties":{"Node name for S&R":"VAEDecode"},"widgets_values":[]},{"id":11,"type":"SaveImage","title":"생성 이미지","pos":[1540,40],"size":[470,500],"flags":{},"order":10,"mode":0,"inputs":[{"name":"images","type":"IMAGE","link":11}],"outputs":[{"name":"images","type":"IMAGE","links":null,"slot_index":0}],"properties":{"Node name for S&R":"SaveImage"},"widgets_values":["MOSS_T4_FFstyle_Heal"]}],"links":[[1,1,0,2,0,"MODEL"],[3,2,0,9,0,"MODEL"],[4,2,1,6,0,"CLIP"],[5,2,1,7,0,"CLIP"],[6,6,0,9,1,"CONDITIONING"],[7,7,0,9,2,"CONDITIONING"],[8,8,0,9,3,"LATENT"],[9,9,0,10,0,"LATENT"],[10,5,0,10,1,"VAE"],[11,10,0,11,0,"IMAGE"],[12,4,0,2,1,"CLIP"]],"groups":[],"config":{},"extra":{"ds":{"scale":0.7,"offset":[60,80]}},"version":0.4}',
}
for filename, source in EXPERIMENTAL_WORKFLOWS.items():
    (workflow_library.parent/filename).write_text(source,encoding='utf-8')


CLOUDFLARED_VERSION = '2026.10.3'
CLOUDFLARED_SHA256 = '053b80b0a23031a038fdbf3c073f59f82b8f1f152aa67971d7383ffafc0dbab0'
CLOUDFLARED = ROOT/'cloudflared'
if not CLOUDFLARED.exists() or hashlib.sha256(CLOUDFLARED.read_bytes()).hexdigest()!=CLOUDFLARED_SHA256:
    binary = fetch(f'https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64')
    if hashlib.sha256(binary).hexdigest()!=CLOUDFLARED_SHA256:
        raise RuntimeError('cloudflared 다운로드의 SHA256이 일치하지 않습니다.')
    CLOUDFLARED.write_bytes(binary)
CLOUDFLARED.chmod(0o755)
(ROOT/'frontend-environment.json').write_text(json.dumps(dict(
    comfy_revision=COMFY_REV,spectrum_revision=SPECTRUM_REV,model_revision=MODEL_REV,lora_revision=LORA_REV,
    cloudflared_version=CLOUDFLARED_VERSION,gpu=gpu,colab_runtime_packages=runtime_packages),indent=2),encoding='utf-8')
print(f'Hugging Face 파일 {len(resources)}개 · Civitai LoRA {civitai_ready}/{len(CIVITAI_LORAS)}개 · 워크플로 · cloudflared 준비 완료')


In [ ]:
#@title 3. ComfyUI 시작 · cloudflared 접속 주소
from concurrent.futures import ThreadPoolExecutor
from threading import Event

TUNNEL_LOG = ROOT/'cloudflared.log'

def stop_process(process):
    if process is None or process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()

def serve_tunnel(stopping):
    tunnel_process = None
    try:
        deadline = time.monotonic()+180
        while not stopping.is_set():
            try:
                with urllib.request.urlopen(LOCAL_URL+'/object_info',timeout=2) as response:
                    object_info = json.load(response)
                break
            except (urllib.error.URLError,TimeoutError):
                if time.monotonic()>deadline:
                    raise RuntimeError('ComfyUI 시작 시간 초과')
                stopping.wait(1)
        else:
            return

        if stopping.is_set():
            return

        missing = {node['type'] for node in workflow['nodes']}-set(object_info)
        if missing:
            raise RuntimeError(f'필수 노드 로딩 실패: {sorted(missing)}')

        with TUNNEL_LOG.open('w') as log:
            tunnel_process = subprocess.Popen(
                [str(CLOUDFLARED),'--no-autoupdate','tunnel','--url',LOCAL_URL,'--protocol','http2'],
                cwd=COMFY,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)

        deadline = time.monotonic()+120
        tunnel_url = None
        while not stopping.is_set():
            if tunnel_process.poll() is not None or (not tunnel_url and time.monotonic()>deadline):
                log = TUNNEL_LOG.read_text(encoding='utf-8',errors='replace')
                raise RuntimeError('cloudflared 연결 실패'+chr(10)+chr(10).join(log.splitlines()[-60:]))
            if tunnel_url is None:
                log = TUNNEL_LOG.read_text(encoding='utf-8',errors='replace')
                match = re.search(r'https://[a-z0-9-]+[.]trycloudflare[.]com',log)
                if match and 'Registered tunnel connection' in log:
                    tunnel_url = match.group(0)
                    display(HTML(f'<h3><a href="{tunnel_url}" target="_blank" rel="noopener noreferrer">ComfyUI 열기</a></h3>'))
                    print('Workflows에서 워크플로를 여세요. 이 셀을 중지하면 서버와 터널이 종료됩니다.',flush=True)
            stopping.wait(1)
    except (OSError, ValueError, RuntimeError) as error:
        print(f'터널 오류: {error}',flush=True)
    finally:
        stop_process(tunnel_process)

print('ComfyUI 포그라운드 실행 · 이 셀을 중지하면 서버가 종료됩니다.',flush=True)
stopping = Event()
with ThreadPoolExecutor(max_workers=1) as executor:
    tunnel = executor.submit(serve_tunnel,stopping)
    try:
        subprocess.run(
            [PY,'-u',str(COMFY/'main.py'),'--listen','127.0.0.1','--port',str(PORT),'--disable-auto-launch'],
            cwd=COMFY,check=True)
    except KeyboardInterrupt:
        print('ComfyUI와 터널을 종료합니다.',flush=True)
    except subprocess.CalledProcessError as error:
        raise RuntimeError(f'ComfyUI 프로세스가 종료됐습니다. 종료 코드 {error.returncode}. 위 서버 로그를 확인하세요.') from error
    finally:
        stopping.set()
        tunnel.result()


In [ ]:
#@title 4. 워크플로 JSON 다운로드
WORKFLOW = 'MOSS_T4_LoRA8_Spectrum.json' #@param ["MOSS_T4_LoRA8_Spectrum.json", "MOSS_T4_NAI5_Style.json", "MOSS_T4_Munji_Style.json", "MOSS_T4_FFstyle_Heal.json"]
from google.colab import files
files.download(str(workflow_library.parent/WORKFLOW))


In [ ]:
#@title 5. ComfyUI · 터널 종료
print('실행 중인 3번 셀의 중지 버튼을 누르면 ComfyUI와 터널이 종료됩니다.')
